In [0]:
%run ../../config

## silver_usageでやりたいこと

- 型変換
- identity_metadata カラムの run_asから email を抽出
- null値削除
- trim(空白削除)
- _重複削除_

In [0]:
# silverテーブルを作成
spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {silver_usage_table_path}
    (
        `record_id` STRING,
        `usage_start_time` TIMESTAMP,
        `usage_end_time` TIMESTAMP,
        `usage_quantity` DOUBLE,
        `sku` STRING,
        `email` STRING,
        `workspace_id` STRING,
        `identity_metadata` STRING,
        _datasource STRING,
        _ingest_timestamp TIMESTAMP
    )
    """
)

# テーブル作り直し
# spark.sql(
#     f"""
#     REPLACE TABLE {silver_usage_table_path} (
#         `record_id` STRING,
#         `usage_start_time` TIMESTAMP,
#         `usage_end_time` TIMESTAMP,
#         `usage_quantity` DOUBLE,
#         `sku` STRING,
#         `email` STRING,
#         `workspace_id` STRING,
#         `identity_metadata` STRING,
#         _datasource STRING,
#         _ingest_timestamp TIMESTAMP
#     )
#     """
# )

## ブロンズテーブルを抽出し、型変換

### 参考）Databricks / Delta 数値型まとめ（用途・推奨中心）

ChatGPTで生成・Perplexityで確認

| 型             | 分類   | 主な用途      | 推奨度     | コメント（理由）                     |
| ------------- | ---- | --------- | ------- | ---------------------------- |
| TINYINT       | 整数   | 小さなフラグ    | △       | 範囲が非常に小さく実務ではほぼ使わない          |
| SMALLINT      | 整数   | 小規模カウント   | △       | 将来増加リスクがあるためあまり使わない          |
| INT           | 整数   | 小規模ID     | ◯       | 件数やIDが増える可能性があるならBIGINTの方が安全 |
| BIGINT (LONG) | 整数   | ID・件数     | ◎       | 将来の増加にも安全で、実務では基本これを使う       |
| FLOAT         | 浮動小数 | 軽量計測値     | ✕       | 精度が低く誤差が出やすい。DOUBLEで代替可能     |
| DOUBLE        | 浮動小数 | 利用量・メトリクス | ◎       | 小数計算の標準。分析用途に最適              |
| DECIMAL(p,s)  | 固定小数 | 金額・課金     | ◎（金額用途） | 誤差が出ないため会計・請求用途で必須           |



In [0]:
df = spark.sql(
    f"""
    SELECT
        record_id,
        CAST(usage_start_time AS TIMESTAMP) AS usage_start_time,
        CAST(usage_end_time AS TIMESTAMP) AS usage_end_time,
        CAST(usage_quantity AS DOUBLE) AS usage_quantity,
        sku,
        identity_metadata,
        workspace_id,
        _datasource,
        _ingest_timestamp
    FROM {bronze_usage_table_path}
    """
)


In [0]:
# 処理後の結果を確認
df.display()

### クレンジング

- trim処理（値の空白除去）
- explodeでJSON展開
- null削除
- 重複削除

空白文字・null値

In [0]:
from pyspark.sql import functions as F

# trim処理
df = (
    df.withColumn("sku", F.trim(F.col("sku")))
    .withColumn("identity_metadata", F.trim(F.col("identity_metadata")))
)

# Null値削除
df = df.filter(F.col("record_id").isNotNull() & F.col("workspace_id").isNotNull())

user JSON を展開（identity_metadataカラムの`run_as.email`をemail列にする）

In [0]:
from pyspark.sql.functions import get_json_object

df = df.withColumn("email", get_json_object("identity_metadata", "$.run_as.email"))

df.display()

重複削除

In [0]:
# 重複削除
# 指定キーの一意性を保証するが、同じevent_idが複数ある時にどれが残るかは分からない
df = df.dropDuplicates(["record_id"])

# 以下で、同じ event_id があったら 最新_ingestを残すこともできる
# from pyspark.sql import functions as F
# from pyspark.sql.window import Window

# w = Window.partitionBy("event_id").orderBy(F.col("_ingest_timestamp").desc())

# df = (
#   df.withColumn("_rn", F.row_number().over(w))
#     .filter(F.col("_rn") == 1)
#     .drop("_rn")
# )

In [0]:
# カラム順序を整列
df = df.select(
    "record_id",
    "usage_start_time",
    "usage_end_time",
    "usage_quantity",
    "sku",
    "email",
    "identity_metadata",
    "workspace_id",
    "_datasource",
    "_ingest_timestamp"
)


シルバーテーブルに書き込む

In [0]:
(
    df.write.format("delta")
    .mode("overwrite")
    .saveAsTable(silver_usage_table_path)
)

# spark.sql(f"optimize {silver_audit_table_path} zorder by (Timestamp)")
display(spark.sql(f"select * from {silver_usage_table_path}"))

In [0]:
%sql
describe history my_lab.handson.silver_audit